# Racial Composition of the Universities

In this notebook, we are manipulating the College Scorecard data to make it compatible to our Census data, as well as calculating the indicators used in other parts of the project.

In [186]:
#setup chunk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from IPython.display import IFrame, Markdown, display
import statsmodels.formula.api as smf

We need to filter the data in the same way in which it is filtered in Glynn's analysis. 

In [187]:
#data cleaning process-- especially making smaller datasets that are more manageable and only include relevant variables

dat = pd.read_csv('../data/Glynn_data/combined_data.csv')
dat.columns

Index(['univ_id', 'univ_name', 'state', 'zip_code', 'percent_pell',
       'completion_rate_fry', 'ugds_white', 'ugds_black', 'ugds_hisp',
       'ugds_asian', 'ugds_aian', 'ugds_nhpi', 'ugds_2mor', 'ugds_nra',
       'ugds_unkn', 'ugds_men', 'ugds_women', 'md_earn_wne_4yr', 'year',
       'hbcu', 'pbi', 'aanapii', 'hsi', 'tribal', 'long', 'all_male',
       'all_female', 'admit_rate', 'lat', 'pct_w_ba_hometown', 'degree_type'],
      dtype='str')

In [188]:
ugds_cols = [colname for colname in dat.columns if colname.startswith('ugds_')]
ugds_cols

['ugds_white',
 'ugds_black',
 'ugds_hisp',
 'ugds_asian',
 'ugds_aian',
 'ugds_nhpi',
 'ugds_2mor',
 'ugds_nra',
 'ugds_unkn',
 'ugds_men',
 'ugds_women']

As part of the filtering, we need to exclude universities that are not part of the 50 US states and universities that are not primarily 4-year institutions

In [189]:
#then narrow only to universities in upper-50 US (no PR or Guam)
dat['state'].value_counts()
non_50 = ['VI', 'AS', 'MP', 'FM', 'PW', 'MH', 'GU', 'PR']
    #virgin islands, american samoa, northern marina islands, federated state of micronesia, palao, guam, puerto rico

dat[~dat['state'].isin(non_50)].nunique() #taking out states not in the 50 states gives us 51 unique states (since the dataset includes DC)
#filtering
dat = dat[~dat['state'].isin(non_50)]

In [190]:
#keep only 4-yr institutions
dat = dat[dat['degree_type']==3]

In [191]:
dat

,univ_id,univ_name,state,zip_code,percent_pell,completion_rate_fry,ugds_white,ugds_black,ugds_hisp,ugds_asian,...,aanapii,hsi,tribal,long,all_male,all_female,admit_rate,lat,pct_w_ba_hometown,degree_type
0,100654,Alabama A & M University,AL,35762,0.6536,0.2678,0.0198,0.8955,0.0110,0.0019,...,0.0,0.0,0.0,-86.568502,0.0,0.0,0.5795,34.783368,13,3
1,100663,University of Alabama at Birmingham,AL,35294-0110,0.3308,0.6442,0.5130,0.2528,0.0711,0.0819,...,0.0,0.0,0.0,-86.799345,0.0,0.0,0.8818,33.505697,15.9300003051757,3
2,100690,Amridge University,AL,36117-3553,0.7769,0.5000,0.2851,0.6623,0.0307,0.0000,...,0.0,0.0,0.0,-86.174010,0.0,0.0,NaN,32.362609,13.2299995422363,3
3,100706,University of Alabama in Huntsville,AL,35899,0.2173,0.6295,0.7102,0.0873,0.0666,0.0389,...,0.0,0.0,0.0,-86.640449,0.0,0.0,0.6857,34.724557,17.6700000762939,3
4,100724,Alabama State University,AL,36104-0271,0.6976,0.2773,0.0155,0.9251,0.0121,0.0015,...,0.0,0.0,0.0,-86.295677,0.0,0.0,0.9755,32.364317,11.8100004196167,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5720,498049,Arizona College of Nursing-Southfield,MI,48033-7157,0.7206,NaN,0.2121,0.6397,0.0808,0.0236,...,0.0,0.0,0.0,-83.260202,0.0,0.0,1.0000,42.481560,10.9499998092651,3
5735,498447,Arizona College of Nursing-Falls Church,VA,22042-4566,NaN,NaN,0.1597,0.5694,0.1528,0.0556,...,0.0,0.0,0.0,-77.218156,0.0,0.0,1.0000,38.860531,10.9499998092651,3
5736,498456,Arizona College of Nursing-Ontario,CA,91761-1201,NaN,NaN,0.1048,0.1333,0.5381,0.1000,...,0.0,0.0,0.0,-117.578071,0.0,0.0,1.0000,34.066760,10.9499998092651,3
5740,498562,Commonwealth University of Pennsylvania,PA,17815,0.3184,0.5356,0.8005,0.0627,0.0638,0.0117,...,0.0,0.0,0.0,-76.447844,0.0,0.0,0.9309,41.007820,13.2200002670288,3


In [192]:
dat = dat[['univ_id','univ_name','state','zip_code','ugds_white',
 'ugds_black',
 'ugds_hisp',
 'ugds_asian',
 'ugds_aian',
 'ugds_nhpi',
 'ugds_2mor',
 'ugds_unkn',
 'ugds_men',
 'ugds_women',
          'lat','long']]

In [193]:
race_cols = ['ugds_white', 'ugds_black', 'ugds_hisp', 'ugds_asian', 'ugds_aian', 
             'ugds_nhpi', 'ugds_2mor']


# mapping column names to labels
#      white, black, hispanic, asian, amarican indian and alaska native, 
#     native hawaiian and pacific islander, two or more races, norace, unknown, 
#     white nonhispanic, blacknonhispanic, asian pacific islander
#

col_labels = {
    'ugds_white': 'White_alone', 
    'ugds_black': 'Black_alone', 
    'ugds_hisp': 'Hispanic_Latino', 
    'ugds_asian': 'Asian_alone', 
    'ugds_aian': 'AIAN_alone', 
    'ugds_nhpi': 'NHPI_alone', 
    'ugds_2mor': 'Two_or_More',
    'ugds_unkn': "Other_alone",
     'ugds_men': "Men",
     'ugds_women': "Women"
}


In [194]:
dat = dat.rename(columns=col_labels)

Now that we have the data we need, we need to create the vectors that will be needed to combine the analysis with the data on census racial composition

In [195]:
dat

,univ_id,univ_name,state,zip_code,White_alone,Black_alone,Hispanic_Latino,Asian_alone,AIAN_alone,NHPI_alone,Two_or_More,Other_alone,Men,Women,lat,long
0,100654,Alabama A & M University,AL,35762,0.0198,0.8955,0.0110,0.0019,0.0025,0.0015,0.0127,0.0435,0.4055,0.5945,34.783368,-86.568502
1,100663,University of Alabama at Birmingham,AL,35294-0110,0.5130,0.2528,0.0711,0.0819,0.0016,0.0005,0.0491,0.0064,0.3752,0.6248,33.505697,-86.799345
2,100690,Amridge University,AL,36117-3553,0.2851,0.6623,0.0307,0.0000,0.0044,0.0044,0.0000,0.0132,0.3640,0.6360,32.362609,-86.174010
3,100706,University of Alabama in Huntsville,AL,35899,0.7102,0.0873,0.0666,0.0389,0.0087,0.0016,0.0465,0.0252,0.5981,0.4019,34.724557,-86.640449
4,100724,Alabama State University,AL,36104-0271,0.0155,0.9251,0.0121,0.0015,0.0021,0.0009,0.0118,0.0088,0.3595,0.6405,32.364317,-86.295677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5720,498049,Arizona College of Nursing-Southfield,MI,48033-7157,0.2121,0.6397,0.0808,0.0236,0.0034,0.0034,0.0370,0.0000,0.0673,0.9327,42.481560,-83.260202
5735,498447,Arizona College of Nursing-Falls Church,VA,22042-4566,0.1597,0.5694,0.1528,0.0556,0.0000,0.0347,0.0278,0.0000,0.0625,0.9375,38.860531,-77.218156
5736,498456,Arizona College of Nursing-Ontario,CA,91761-1201,0.1048,0.1333,0.5381,0.1000,0.0048,0.0143,0.1048,0.0000,0.1095,0.8905,34.066760,-117.578071
5740,498562,Commonwealth University of Pennsylvania,PA,17815,0.8005,0.0627,0.0638,0.0117,0.0037,0.0008,0.0254,0.0278,0.3914,0.6086,41.007820,-76.447844


When looking at the data, we can see that the geographic data for each university is regarding their zipcode. However, our census data is done via county or county-equivalents. As such, we need to maake these two dataframes compatible using a dictionary/crosswalk from ZIP codes to County IDs.

In [196]:
zip_dic = pd.read_csv('../data/zip_county_092022.csv')

In [197]:
print(zip_dic.columns.tolist())

['ZIP', 'COUNTY', 'USPS_ZIP_PREF_CITY', 'USPS_ZIP_PREF_STATE', 'RES_RATIO', 'BUS_RATIO', 'OTH_RATIO', 'TOT_RATIO']


In [198]:
# 1. Sort the crosswalk so the highest ratio is at the top for each ZIP
zip_dic_sorted = zip_dic.sort_values(by=['ZIP', 'RES_RATIO'], ascending=[True, False])

# 2. Drop duplicates, keeping only the first (which is now the highest ratio)
zip_dic_primary = zip_dic_sorted.drop_duplicates(subset=['ZIP'], keep='first')

In [199]:
# 1. Force both ZIP columns to be 5-digit strings
# This prevents '02138' (string) failing to match 2138 (number)
dat['zip_code'] = dat['zip_code'].astype(str).str.split('-').str[0].str.split('.').str[0].str.zfill(5)

zip_dic_primary['ZIP'] = zip_dic_primary['ZIP'].astype(str).str.zfill(5)
# 2. Merge the data
# We keep all rows from 'dat' and bring in the county from 'zip_dic'
dat = dat.merge(zip_dic_primary[['ZIP', 'COUNTY']], 
                left_on='zip_code', 
                right_on='ZIP', 
                how='left')

# 3. Clean up
dat = dat.drop(columns=['ZIP'])

print("Success! Here is a sample:")
print(dat[['univ_name', 'zip_code', 'COUNTY']].head())

Success! Here is a sample:
                             univ_name zip_code  COUNTY
0             Alabama A & M University    35762  1089.0
1  University of Alabama at Birmingham    35294  1073.0
2                   Amridge University    36117  1101.0
3  University of Alabama in Huntsville    35899  1089.0
4             Alabama State University    36104  1101.0


In [200]:
# 1. Fill NaNs with a placeholder or handle them
# 2. Convert to integer to get rid of the '.0'
# 3. Convert to string and pad with 'zfill' to ensure it is 5 digits long
dat['COUNTY'] = dat['COUNTY'].fillna(0).astype(int).astype(str).str.zfill(5)
import numpy as np
dat['COUNTY'] = dat['COUNTY'].replace('00000', np.nan)

print(dat[['univ_name', 'COUNTY']].head())

                             univ_name COUNTY
0             Alabama A & M University  01089
1  University of Alabama at Birmingham  01073
2                   Amridge University  01101
3  University of Alabama in Huntsville  01089
4             Alabama State University  01101


In [201]:
dat

,univ_id,univ_name,state,zip_code,White_alone,Black_alone,Hispanic_Latino,Asian_alone,AIAN_alone,NHPI_alone,Two_or_More,Other_alone,Men,Women,lat,long,COUNTY
0,100654,Alabama A & M University,AL,35762,0.0198,0.8955,0.0110,0.0019,0.0025,0.0015,0.0127,0.0435,0.4055,0.5945,34.783368,-86.568502,01089
1,100663,University of Alabama at Birmingham,AL,35294,0.5130,0.2528,0.0711,0.0819,0.0016,0.0005,0.0491,0.0064,0.3752,0.6248,33.505697,-86.799345,01073
2,100690,Amridge University,AL,36117,0.2851,0.6623,0.0307,0.0000,0.0044,0.0044,0.0000,0.0132,0.3640,0.6360,32.362609,-86.174010,01101
3,100706,University of Alabama in Huntsville,AL,35899,0.7102,0.0873,0.0666,0.0389,0.0087,0.0016,0.0465,0.0252,0.5981,0.4019,34.724557,-86.640449,01089
4,100724,Alabama State University,AL,36104,0.0155,0.9251,0.0121,0.0015,0.0021,0.0009,0.0118,0.0088,0.3595,0.6405,32.364317,-86.295677,01101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1893,498049,Arizona College of Nursing-Southfield,MI,48033,0.2121,0.6397,0.0808,0.0236,0.0034,0.0034,0.0370,0.0000,0.0673,0.9327,42.481560,-83.260202,26125
1894,498447,Arizona College of Nursing-Falls Church,VA,22042,0.1597,0.5694,0.1528,0.0556,0.0000,0.0347,0.0278,0.0000,0.0625,0.9375,38.860531,-77.218156,51059
1895,498456,Arizona College of Nursing-Ontario,CA,91761,0.1048,0.1333,0.5381,0.1000,0.0048,0.0143,0.1048,0.0000,0.1095,0.8905,34.066760,-117.578071,06071
1896,498562,Commonwealth University of Pennsylvania,PA,17815,0.8005,0.0627,0.0638,0.0117,0.0037,0.0008,0.0254,0.0278,0.3914,0.6086,41.007820,-76.447844,42037


In [202]:
race_columns = ['White_alone', 'Hispanic_Latino', 'Black_alone', 'AIAN_alone', 
                'Asian_alone', 'NHPI_alone', 'Other_alone', 'Two_or_More']


In [203]:
# This creates a list of indices and names for the missing rows
missing = dat[dat['COUNTY'].isna()][['univ_name']]
print(missing)

                                              univ_name
5                             The University of Alabama
482                                       Berea College
521                           Nicholls State University
528               Southern University and A & M College
582                                      Babson College
641                          Westfield State University
1023                                   Davidson College
1050         North Carolina State University at Raleigh
1094               University of Cincinnati-Main Campus
1216                                Duquesne University
1397                 Tennessee Technological University
1447                      Southern Methodist University
1532                          Virginia State University
1623             California State University-San Marcos
1671                     Southern University Law Center
1834  The University of Tennessee Health Science Center


There are a couple of universities where the crosswalk converter did not work to identify their respectiev county. Since there are only 16 universities, we can manually add the county FIPS to each of them to avoid having to take them out from the data.

In [204]:
# Target by index (the number on the far left of your printout)
dat.loc[5, 'COUNTY'] = '01125' 
dat.loc[482, 'COUNTY'] = '21151'
dat.loc[521, 'COUNTY'] = '22057' 
dat.loc[528, 'COUNTY'] = '22033'
dat.loc[582, 'COUNTY'] = '25021' 
dat.loc[641, 'COUNTY'] = '25013'
dat.loc[1023, 'COUNTY'] = '37119' 
dat.loc[1050, 'COUNTY'] = '37183'
dat.loc[1094, 'COUNTY'] = '39061' 
dat.loc[1216, 'COUNTY'] = '42003'
dat.loc[1397, 'COUNTY'] = '47141' 
dat.loc[1447, 'COUNTY'] = '48113'
dat.loc[1532, 'COUNTY'] = '51041' 
dat.loc[1623, 'COUNTY'] = '06073'
dat.loc[1671, 'COUNTY'] = '22033' 
dat.loc[1834, 'COUNTY'] = '47157'

In [205]:
# Drop rows where the racial data is missing
dat = dat.dropna(subset=['White_alone'])

# Verify that the count is now zero for those columns
print(dat.isna().sum())

univ_id            0
univ_name          0
state              0
zip_code           0
White_alone        0
Black_alone        0
Hispanic_Latino    0
Asian_alone        0
AIAN_alone         0
NHPI_alone         0
Two_or_More        0
Other_alone        0
Men                0
Women              0
lat                0
long               0
COUNTY             0
dtype: int64


In [206]:
dat_proportions = dat[race_columns]

Now that we have all the data as needed, we want to calculate the indicators for each of the university. As such, we need to store the racial compositions as vectors.

In [207]:
# Create a new vector dataframe
dat_vectors = dat[['COUNTY', 'univ_id','univ_name']].copy()

# Store these proportions as a vector
dat_vectors['race_proportion_vector'] = dat_proportions.values.tolist()

dat_vectors.head()

,COUNTY,univ_id,univ_name,race_proportion_vector
0,01089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015..."
1,01073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005..."
2,01101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ..."
3,01089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001..."
4,01101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000..."


In [208]:
# Calculate the sum of squares for each row in the proportions dataframe
# We square the numbers first (** 2) and then sum them horizontally (axis=1)
dat_vectors['diversity_index'] = (dat_proportions ** 2).sum(axis=1).round(4)

dat_vectors.head()

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index
0,01089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045
1,01073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413
2,01101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211
3,01089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208
4,01101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564


In [209]:
# Store the merged data to manipulate with the census data later
dat_vectors.to_csv('../data/racial_comp_univ.csv', index=False)

In [210]:
dat_vectors.head(6)

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index
0,01089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045
1,01073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413
2,01101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211
3,01089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208
4,01101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564
5,01125,100751,The University of Alabama,"[0.7336, 0.0577, 0.1072, 0.0036, 0.0146, 0.001...",0.5559


In [215]:
national_proportions = [0.1952, 0.5728, 0.1193, 0.0067, 0.0586, 0.0019, 0.0051, 0.0405]
national_vector = np.array(national_proportions)
national_vector

array([0.1952, 0.5728, 0.1193, 0.0067, 0.0586, 0.0019, 0.0051, 0.0405])

Now that we have all the relevant vectors, we want to calculate the similarity between them and their surrounding community at a county, state, and federal level. 

In [222]:
state_vector_df = pd.read_csv('../data/racial_comp_state.csv')

In [147]:
county_vector_df = pd.read_csv('../data/racial_comp_county.csv')

In [211]:
university_vector_df = pd.read_csv('../data/racial_comp_univ.csv')

In [250]:
# Create a dictionary for quick lookup of state vectors
state_vector_lookup = state_vector_df['state_vector'].to_dict()

# Add a column to university_vector_df that identifies the state FIPS
university_vector_df['State_FIPS'] = (
    university_vector_df['COUNTY']
    .astype(str)
    .str.zfill(5)
    .str[:2]
)

In [255]:
university_vector_df.head()

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index,State_FIPS,similarity_to_national,similarity_to_state
0,1089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045,01,0.211625,0.388022
1,1073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413,01,0.490277,0.349466
2,1101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211,01,0.339893,0.413371
3,1089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208,01,0.427433,0.210909
4,1101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564,01,0.210471,0.388572


In [256]:
import ast
from sklearn.metrics.pairwise import cosine_similarity

# Parse the string vectors into actual lists first
parsed = university_vector_df['race_proportion_vector'].apply(ast.literal_eval)

# Now convert to 2D float matrix
X = np.array(parsed.tolist(), dtype=float)

# State similarity
Y = np.array(national_vector, dtype=float).reshape(1, -1)
university_vector_df['similarity_to_national'] = cosine_similarity(X, Y).flatten()

In [257]:
university_vector_df

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index,State_FIPS,similarity_to_national,similarity_to_state
0,1089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045,01,0.211625,0.388022
1,1073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413,01,0.490277,0.349466
2,1101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211,01,0.339893,0.413371
3,1089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208,01,0.427433,0.210909
4,1101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564,01,0.210471,0.388572
...,...,...,...,...,...,...,...,...
1892,26125,498049,Arizona College of Nursing-Southfield,"[0.2121, 0.0808, 0.6397, 0.0034, 0.0236, 0.003...",0.4627,26,0.395171,0.279019
1893,51059,498447,Arizona College of Nursing-Falls Church,"[0.1597, 0.1528, 0.5694, 0.0, 0.0556, 0.0347, ...",0.3781,51,0.500408,0.437796
1894,6071,498456,Arizona College of Nursing-Ontario,"[0.1048, 0.5381, 0.1333, 0.0048, 0.1, 0.0143, ...",0.3395,06,0.980461,0.990352
1895,42037,498562,Commonwealth University of Pennsylvania,"[0.8005, 0.0638, 0.0627, 0.0037, 0.0117, 0.000...",0.6504,42,0.403706,0.774565


In [258]:
state_vector_df

,State_FIPS,Hispanic_Latino,White_alone,Black_alone,AIAN_alone,Asian_alone,NHPI_alone,Other_alone,Two_or_More,diversity_index,state_vector
0,1,0.0526,0.6312,0.2564,0.0046,0.0151,0.0005,0.0029,0.0367,0.4685,"[0.0526, 0.6312, 0.2564, 0.0046, 0.0151, 0.000..."
1,2,0.0679,0.5751,0.0283,0.1484,0.0592,0.0170,0.0062,0.0978,0.3716,"[0.0679, 0.5751, 0.0283, 0.1484, 0.0592, 0.017..."
2,4,0.3065,0.5337,0.0443,0.0369,0.0348,0.0020,0.0044,0.0373,0.3847,"[0.3065, 0.5337, 0.0443, 0.0369, 0.0348, 0.002..."
3,5,0.0853,0.6852,0.1494,0.0068,0.0170,0.0047,0.0027,0.0489,0.5019,"[0.0853, 0.6852, 0.1494, 0.0068, 0.017, 0.0047..."
4,6,0.3940,0.3469,0.0536,0.0039,0.1512,0.0035,0.0057,0.0412,0.3031,"[0.394, 0.3469, 0.0536, 0.0039, 0.1512, 0.0035..."
5,8,0.2188,0.6513,0.0383,0.0058,0.0338,0.0016,0.0051,0.0452,0.4768,"[0.2188, 0.6513, 0.0383, 0.0058, 0.0338, 0.001..."
6,9,0.1729,0.6321,0.1001,0.0018,0.0473,0.0003,0.0075,0.0382,0.4432,"[0.1729, 0.6321, 0.1001, 0.0018, 0.0473, 0.000..."
7,10,0.1053,0.5857,0.2151,0.0025,0.0428,0.0003,0.0046,0.0435,0.4042,"[0.1053, 0.5857, 0.2151, 0.0025, 0.0428, 0.000..."
8,11,0.1126,0.3796,0.4091,0.0019,0.0481,0.0005,0.0054,0.0428,0.3283,"[0.1126, 0.3796, 0.4091, 0.0019, 0.0481, 0.000..."
9,12,0.2645,0.5154,0.1452,0.0020,0.0292,0.0005,0.0064,0.0368,0.3589,"[0.2645, 0.5154, 0.1452, 0.002, 0.0292, 0.0005..."


In [260]:
def calculate_similarity(vec1, vec2):
    # Reshape vectors because sklearn expects 2D arrays
    v1 = np.array(vec1).reshape(1, -1)
    v2 = np.array(vec2).reshape(1, -1)
    return round(cosine_similarity(v1, v2)[0][0], 4)

In [263]:
race_cols = ['Hispanic_Latino', 'White_alone', 'Black_alone', 'AIAN_alone', 
             'Asian_alone', 'NHPI_alone', 'Other_alone', 'Two_or_More']

state_lookup_by_fips = {
    str(int(row['State_FIPS'])).zfill(2): row[race_cols].values.astype(float)
    for _, row in state_vector_df.iterrows()
}

state_vectors = university_vector_df['State_FIPS'].map(state_lookup_by_fips)
print("Unmatched:", state_vectors.isna().sum())

valid_mask = state_vectors.notna()
S = np.vstack(state_vectors[valid_mask].values).astype(float)
X_valid = X[valid_mask.values]

sims = (X_valid * S).sum(axis=1) / (
    np.linalg.norm(X_valid, axis=1) * np.linalg.norm(S, axis=1)
)

university_vector_df['similarity_to_state'] = np.nan
university_vector_df.loc[valid_mask, 'similarity_to_state'] = sims

print(university_vector_df[['univ_name', 'State_FIPS', 'similarity_to_state']].head())

state_lookup_by_fips = {
    str(int(row['State_FIPS'])).zfill(2): row[race_cols].values.astype(float)
    for _, row in state_vector_df.iterrows()
}

state_vectors = university_vector_df['State_FIPS'].map(state_lookup_by_fips)
print("Unmatched:", state_vectors.isna().sum())

valid_mask = state_vectors.notna()
S = np.vstack(state_vectors[valid_mask].values).astype(float)
X_valid = X[valid_mask.values]

sims = (X_valid * S).sum(axis=1) / (
    np.linalg.norm(X_valid, axis=1) * np.linalg.norm(S, axis=1)
)

university_vector_df['similarity_to_state'] = np.nan
university_vector_df.loc[valid_mask, 'similarity_to_state'] = sims

print(university_vector_df[['univ_name', 'State_FIPS', 'similarity_to_state']].head())

Unmatched: 0
                             univ_name State_FIPS  similarity_to_state
0             Alabama A & M University         01             0.388022
1  University of Alabama at Birmingham         01             0.349466
2                   Amridge University         01             0.413371
3  University of Alabama in Huntsville         01             0.210909
4             Alabama State University         01             0.388572
Unmatched: 0
                             univ_name State_FIPS  similarity_to_state
0             Alabama A & M University         01             0.388022
1  University of Alabama at Birmingham         01             0.349466
2                   Amridge University         01             0.413371
3  University of Alabama in Huntsville         01             0.210909
4             Alabama State University         01             0.388572


In [264]:
university_vector_df

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index,State_FIPS,similarity_to_national,similarity_to_state
0,1089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045,01,0.211625,0.388022
1,1073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413,01,0.490277,0.349466
2,1101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211,01,0.339893,0.413371
3,1089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208,01,0.427433,0.210909
4,1101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564,01,0.210471,0.388572
...,...,...,...,...,...,...,...,...
1892,26125,498049,Arizona College of Nursing-Southfield,"[0.2121, 0.0808, 0.6397, 0.0034, 0.0236, 0.003...",0.4627,26,0.395171,0.315655
1893,51059,498447,Arizona College of Nursing-Falls Church,"[0.1597, 0.1528, 0.5694, 0.0, 0.0556, 0.0347, ...",0.3781,51,0.500408,0.558033
1894,6071,498456,Arizona College of Nursing-Ontario,"[0.1048, 0.5381, 0.1333, 0.0048, 0.1, 0.0143, ...",0.3395,06,0.980461,0.793736
1895,42037,498562,Commonwealth University of Pennsylvania,"[0.8005, 0.0638, 0.0627, 0.0037, 0.0117, 0.000...",0.6504,42,0.403706,0.198049


In [266]:
print(county_vector_df.columns.tolist())

['STATEFP', 'COUNTYFP', 'COUNTYNS', 'AFFGEOID', 'GEOID', 'NAME', 'NAMELSAD', 'STUSPS', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'geometry', 'GEO_ID', 'County_Name', 'race_proportion_vector', 'diversity_index', 'State_FIPS', 'similarity_to_national', 'similarity_to_state', 'FIPS']


In [267]:
print(county_vector_df[['FIPS', 'GEOID']].head())
print(county_vector_df['FIPS'].dtype)

    FIPS  GEOID
0   1059   1059
1   6057   6057
2  26031  26031
3  29119  29119
4  31157  31157
int64


In [269]:
import ast

# Build lookup parsing the string vector
county_lookup = {
    row['FIPS']: np.array(ast.literal_eval(row['race_proportion_vector']))
    for _, row in county_vector_df.iterrows()
}

county_vectors = university_vector_df['COUNTY'].map(county_lookup)
print("Unmatched:", county_vectors.isna().sum())

zero_vector_mask = (X == 0).all(axis=1)

valid_mask = county_vectors.notna() & ~zero_vector_mask
S_county = np.vstack(county_vectors[valid_mask].values).astype(float)
X_valid = X[valid_mask.values]

sims = (X_valid * S_county).sum(axis=1) / (
    np.linalg.norm(X_valid, axis=1) * np.linalg.norm(S_county, axis=1)
)

university_vector_df['similarity_to_county'] = np.nan
university_vector_df.loc[valid_mask, 'similarity_to_county'] = sims

print(university_vector_df[['univ_name', 'COUNTY', 'similarity_to_county']].head())

Unmatched: 0
                             univ_name  COUNTY  similarity_to_county
0             Alabama A & M University    1089              0.369349
1  University of Alabama at Birmingham    1073              0.452353
2                   Amridge University    1101              0.844064
3  University of Alabama in Huntsville    1089              0.231457
4             Alabama State University    1101              0.874359


In [270]:
university_vector_df

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index,State_FIPS,similarity_to_national,similarity_to_state,similarity_to_county
0,1089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045,01,0.211625,0.388022,0.369349
1,1073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413,01,0.490277,0.349466,0.452353
2,1101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211,01,0.339893,0.413371,0.844064
3,1089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208,01,0.427433,0.210909,0.231457
4,1101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564,01,0.210471,0.388572,0.874359
...,...,...,...,...,...,...,...,...,...
1892,26125,498049,Arizona College of Nursing-Southfield,"[0.2121, 0.0808, 0.6397, 0.0034, 0.0236, 0.003...",0.4627,26,0.395171,0.315655,0.319029
1893,51059,498447,Arizona College of Nursing-Falls Church,"[0.1597, 0.1528, 0.5694, 0.0, 0.0556, 0.0347, ...",0.3781,51,0.500408,0.558033,0.489030
1894,6071,498456,Arizona College of Nursing-Ontario,"[0.1048, 0.5381, 0.1333, 0.0048, 0.1, 0.0143, ...",0.3395,06,0.980461,0.793736,0.615397
1895,42037,498562,Commonwealth University of Pennsylvania,"[0.8005, 0.0638, 0.0627, 0.0037, 0.0117, 0.000...",0.6504,42,0.403706,0.198049,0.117546


In [273]:
university_vector_df.to_csv('../data/university_similarity.csv', index=False)

Now we have all the diverity indicators for each school, including the Herfindel diversity index and the similarity scores to their surrounding communities.